In [63]:
import os
import pandas as pd
from datetime import datetime
import pytz
import unicodedata
from rapidfuzz import process, fuzz

# === Date & Paths ===
eastern = pytz.timezone("US/Eastern")
today = datetime.now(eastern).date()
today_str = today.isoformat()

paths = {
    "spreads": f"../data/novig-odds/novig_spreads_{today_str}.csv",
    "pitchers": f"../data/starting-pitchers/starting_pitchers_{today_str}.csv",
    "player_pitching": f"../data/player_pitching/player_pitching_{today_str}.csv",
    "standings": f"../data/standings/standings_{today_str}.csv",
    "team_batting": f"../data/team_batting/team_batting_{today_str}.csv",
    "team_pitching": f"../data/team_pitching/team_pitching_{today_str}.csv"
}

# === Load Data ===
for label, path in paths.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing required file: {path}")

spreads_df = pd.read_csv(paths["spreads"])
pitchers_df = pd.read_csv(paths["pitchers"])
pitching_df = pd.read_csv(paths["player_pitching"])
standings_df = pd.read_csv(paths["standings"])
batting_df = pd.read_csv(paths["team_batting"])
team_pitching_df = pd.read_csv(paths["team_pitching"])

# === Normalize
def normalize_str(s):
    if not isinstance(s, str): return ""
    return unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('utf-8').strip().lower()

# === Clean Names
pitching_df['clean_name'] = pitching_df['Player'].apply(normalize_str)

# === Team Abbr Map
team_name_map = {
    "ATL": "Atlanta Braves", "PHI": "Philadelphia Phillies", "OAK": "Athletics",
    "TOR": "Toronto Blue Jays", "TB": "Tampa Bay Rays", "HOU": "Houston Astros",
    "WAS": "Washington Nationals", "SEA": "Seattle Mariners", "CIN": "Cincinnati Reds",
    "CHC": "Chicago Cubs", "MIL": "Milwaukee Brewers", "CWS": "Chicago White Sox",
    "BAL": "Baltimore Orioles", "LAA": "Los Angeles Angels", "CLE": "Cleveland Guardians",
    "COL": "Colorado Rockies", "NYM": "New York Mets", "SF": "San Francisco Giants",
    "MIA": "Miami Marlins", "BOS": "Boston Red Sox", "STL": "St. Louis Cardinals",
    "TEX": "Texas Rangers", "DET": "Detroit Tigers", "KC": "Kansas City Royals",
    "AZ": "Arizona Diamondbacks", "PIT": "Pittsburgh Pirates", "SD": "San Diego Padres",
    "MIN": "Minnesota Twins", "NYY": "New York Yankees", "LAD": "Los Angeles Dodgers"
}

# === Pitcher Matching
def get_pitcher(team_abbr):
    full_name = team_name_map.get(team_abbr.upper(), None)
    if not full_name:
        return "Unknown"
    home_match = pitchers_df[pitchers_df['Home Team'].str.lower().str.strip() == full_name.lower()]
    away_match = pitchers_df[pitchers_df['Away Team'].str.lower().str.strip() == full_name.lower()]
    if not home_match.empty:
        return home_match['Home Starter'].values[0]
    elif not away_match.empty:
        return away_match['Away Starter'].values[0]
    return "TBD"

spreads_df['fav_pitcher'] = spreads_df['fav_team'].apply(get_pitcher)
spreads_df['dog_pitcher'] = spreads_df['dog_team'].apply(get_pitcher)

# === Fuzzy Match Pitcher Stats
def fuzzy_match_pitcher(name, choices, threshold=80):
    match = process.extractOne(normalize_str(name), choices, scorer=fuzz.ratio, score_cutoff=threshold)
    return match[0] if match else None

def get_pitcher_stats(player):
    norm_name = normalize_str(player)
    row = pitching_df[pitching_df['clean_name'] == norm_name]
    if row.empty:
        match_name = fuzzy_match_pitcher(player, pitching_df['clean_name'].tolist())
        if match_name:
            row = pitching_df[pitching_df['clean_name'] == match_name]
    if row.empty:
        return pd.Series([None]*5)
    row = row.iloc[0]
    return pd.Series([row.get("ERA"), row.get("FIP"), row.get("WHIP"), row.get("SO9"), row.get("BB9"), row.get("ERA+")])

spreads_df[['fav_pitcher_era', 'fav_pitcher_fip', 'fav_pitcher_whip', 'fav_pitcher_so9', 'fav_pitcher_bb9', "fav_pitcher_era+"]] = spreads_df['fav_pitcher'].apply(get_pitcher_stats)
spreads_df[['dog_pitcher_era', 'dog_pitcher_fip', 'dog_pitcher_whip', 'dog_pitcher_so9', 'dog_pitcher_bb9', "dog_pitcher_era+"]] = spreads_df['dog_pitcher'].apply(get_pitcher_stats)

# === Empty Score Columns
spreads_df['fav_score'] = None
spreads_df['dog_score'] = None
spreads_df['home_team'] = spreads_df.apply(lambda row: row['fav_team'] if row['home_favorite'] == 1 else row['dog_team'], axis=1)
spreads_df['away_team'] = spreads_df.apply(lambda row: row['dog_team'] if row['home_favorite'] == 1 else row['fav_team'], axis=1)
spreads_df['home_score'] = None
spreads_df['away_score'] = None

# === Normalize and Join Team Stats
for df in [standings_df, batting_df, team_pitching_df]:
    df['team_clean'] = df['Tm'].apply(normalize_str)

def get_team_stat(team_abbr, df, column):
    full = team_name_map.get(team_abbr.upper(), "")
    clean = normalize_str(full)
    row = df[df['team_clean'] == clean]
    return row[column].values[0] if not row.empty else None

# Add team stats
spreads_df['fav_win_pct'] = spreads_df['fav_team'].apply(lambda abbr: get_team_stat(abbr, standings_df, 'W-L%'))
spreads_df['dog_win_pct'] = spreads_df['dog_team'].apply(lambda abbr: get_team_stat(abbr, standings_df, 'W-L%'))
spreads_df['fav_ba'] = spreads_df['fav_team'].apply(lambda abbr: get_team_stat(abbr, batting_df, 'BA'))
spreads_df['dog_ba'] = spreads_df['dog_team'].apply(lambda abbr: get_team_stat(abbr, batting_df, 'BA'))
spreads_df['fav_era'] = spreads_df['fav_team'].apply(lambda abbr: get_team_stat(abbr, team_pitching_df, 'ERA'))
spreads_df['dog_era'] = spreads_df['dog_team'].apply(lambda abbr: get_team_stat(abbr, team_pitching_df, 'ERA'))

# Ensure lowercase headers
standings_df.columns = [col.lower() for col in standings_df.columns]


ordered_cols = [
    'game_time_est', 'fav_team', 'fav_score', 'dog_team', 'dog_score',
    'home_team', 'home_score', 'away_team', 'away_score',
    'fav_line', 'dog_line', 'fav_price', 'dog_price',
    'fav_price_american', 'dog_price_american', 'home_favorite', 'market',
    'fav_pitcher', 'dog_pitcher',
    'fav_pitcher_era', 'dog_pitcher_era',
    'fav_pitcher_fip', 'dog_pitcher_fip',
    'fav_pitcher_whip', 'dog_pitcher_whip',
    'fav_pitcher_so9', 'dog_pitcher_so9',
    'fav_pitcher_bb9', 'dog_pitcher_bb9',
    'fav_pitcher_era+', 'dog_pitcher_era+',
    'fav_win_pct', 'dog_win_pct',
    'fav_ba', 'dog_ba',
    'fav_era', 'dog_era'
]

spreads_df = spreads_df[ordered_cols]

# === Show Result
print("✅ Final enriched training set (WAR excluded):")
pd.set_option('display.max_columns', None)
pd.set_option('display.expand_frame_repr', False)
display(spreads_df)

# === Save Final DataFrame to CSV ===
output_path = os.path.join("training-set", f"training_set_{today_str}.csv")
os.makedirs(os.path.dirname(output_path), exist_ok=True)
spreads_df.to_csv(output_path, index=False)
print(f"✅ Enriched spreads saved to {output_path}")

✅ Final enriched training set (WAR excluded):


,game_time_est,fav_team,fav_score,dog_team,dog_score,home_team,home_score,away_team,away_score,fav_line,dog_line,fav_price,dog_price,fav_price_american,dog_price_american,home_favorite,market,fav_pitcher,dog_pitcher,fav_pitcher_era,dog_pitcher_era,fav_pitcher_fip,dog_pitcher_fip,fav_pitcher_whip,dog_pitcher_whip,fav_pitcher_so9,dog_pitcher_so9,fav_pitcher_bb9,dog_pitcher_bb9,fav_pitcher_era+,dog_pitcher_era+,fav_win_pct,dog_win_pct,fav_ba,dog_ba,fav_era,dog_era
0,2025-05-29T21:40:00-04:00,SEA,None,WAS,None,SEA,None,WAS,None,-1.5,1.5,0.347,0.671,188,-204,1,SEA -1.5,Emerson Hancock,MacKenzie Gore,5.95,3.47,5.36,2.81,1.653,1.251,6.4,13.4,2.7,3.0,61.0,115.0,0.556,0.455,.238,.241,3.74,5.04
1,2025-05-29T19:07:00-04:00,TOR,None,OAK,None,TOR,None,OAK,None,-1.5,1.5,0.434,0.585,130,-141,1,TOR -1.5,José Berríos,Jacob Lopez,4.22,2.57,4.71,3.71,1.359,1.571,7.9,10.9,3.7,5.1,95.0,164.0,0.491,0.411,.244,.255,4.00,5.39
2,2025-05-29T20:10:00-04:00,HOU,None,TB,None,HOU,None,TB,None,-1.5,1.5,0.348,0.673,187,-206,1,HOU -1.5,Ryan Gusto,Shane Baz,4.58,4.94,4.57,4.92,1.528,1.390,9.7,8.6,4.1,3.6,86.0,78.0,0.545,0.509,.255,.245,3.37,3.58
3,2025-05-29T18:45:00-04:00,ATL,None,PHI,None,PHI,None,ATL,None,-1.5,1.5,0.392,0.620,155,-163,0,PHI +1.5,AJ Smith-Shawver,Cristopher Sánchez,3.67,3.17,3.55,3.44,1.416,1.259,8.9,10.8,4.3,3.3,111.0,130.0,0.472,0.648,.246,.259,3.71,3.65


✅ Enriched spreads saved to training-set/training_set_2025-05-29.csv
